# 도자기 육안조사 - Colab 시작 노트북

MVP 순서: **완전/파편 분류 + 유약 유무 먼저 → 문양/시대/색상 분류 → 그릇 형태 측정**

이 노트북은 1단계(완전/파편, 유약 유무)까지 다룹니다. 유물 분리는 `heritage_corrison`의
Grounding DINO + SAM2를 재사용하고, 텍스트 프롬프트만 도자기용으로 바꿉니다.

**실행 전에**: 런타임 > 런타임 유형 변경 > 하드웨어 가속기 = GPU(T4) 로 설정하세요.

## 1. Drive 마운트 + 기존 레포 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/kt_bigp'
HERITAGE_REPO = os.path.join(PROJECT_ROOT, 'heritage_corrison')
POTTERY_DIR = os.path.join(PROJECT_ROOT, 'pottery_analysis')
os.makedirs(POTTERY_DIR, exist_ok=True)

# heritage_corrison이 이 경로에 없으면 여기서 clone
if not os.path.exists(HERITAGE_REPO):
    %cd {PROJECT_ROOT}
    !git clone https://github.com/Jodorong/heritage_corrison.git
else:
    print('heritage_corrison 이미 존재:', HERITAGE_REPO)

## 2. 환경 구성

In [ ]:
%cd {HERITAGE_REPO}
!pip install -r requirements.txt --quiet
!pip install scipy --quiet

SAM2_DIR = os.path.join(HERITAGE_REPO, 'sam2_repo')
if not os.path.exists(SAM2_DIR):
    !git clone https://github.com/facebookresearch/sam2.git {SAM2_DIR}
else:
    print('sam2_repo 폴더는 이미 존재 (Drive에 저장됨)')

# 중요: Drive에 폴더가 남아있어도, Colab 런타임은 세션마다 설치된
# 패키지가 초기화된다. 그래서 폴더 존재 여부와 상관없이
# pip install은 매 세션 다시 실행해야 한다 (git clone만 스킵 가능).
!pip install -e {SAM2_DIR} --quiet

## 3. 도자기 분석 모듈 저장
아래 두 셀은 `glaze_detector.py`, `completeness_classifier.py`를 Drive의 `pottery_analysis/`에 저장합니다.
한 번 실행해두면 다음 세션부터는 이 셀 없이 바로 import만 하면 됩니다.

In [ ]:
%%writefile {POTTERY_DIR}/glaze_detector.py
"""
glaze_detector.py

도자기 표면의 유약(광택) 유무를 하이라이트 반사 특성으로 판별하는 시작 코드.

접근 방식
---------
유약을 바른 도자기 표면은 빛을 정반사(specular reflection)해서
좋고 밝은 하이라이트가 생긴다. 무유(유약 없음) 표면은 난반사가 많아
밝기가 완만하게 퍼진다. 그래서:

1. 밝기(V 채널) 중 상위 퍼센타일 이상인 픽셀 비율 (하이라이트 면적)
2. 그 하이라이트 영역의 "뿠족함" (밝기 분포의 첨도, kurtosis)
3. 채도(S 채널) 표준편차 - 유광 표면은 하이라이트 부분에서 채도가 급격히 낮아지는 경향

를 특징으로 뿑아서 규칙 기반으로 1차 분류한다.

주의 - 이건 최종 분류기가 아니라 시작점이다
-------------------------------------------
아직 라벨링된 도자기 사진이 없어서, 아래 THRESHOLD 값들은
전부 임시값(placeholder)이다. 실제 유광/무유 도자기 사진을
10~20장씩만 모아도 이 임계값을 실측 분포로 바로 교체해야 한다.
"""

from dataclasses import dataclass

import cv2
import numpy as np
from scipy.stats import kurtosis

THRESHOLDS = {
    "highlight_percentile": 95,
    "highlight_area_ratio_min": 0.005,
    "highlight_kurtosis_min": 3.0,
    "saturation_std_min": 15.0,
}


@dataclass
class GlazeEstimate:
    has_glaze_guess: bool
    confidence_label: str
    highlight_area_ratio: float
    highlight_kurtosis: float
    saturation_std: float
    note: str


def estimate_glaze(image_bgr: np.ndarray, object_mask: np.ndarray) -> GlazeEstimate:
    binary_mask = np.asarray(object_mask, dtype=bool)
    if binary_mask.sum() == 0:
        raise ValueError("object_mask에 유효한 픽셀이 없습니다.")

    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    v_channel = hsv[..., 2][binary_mask].astype(np.float32)
    s_channel = hsv[..., 1][binary_mask].astype(np.float32)

    highlight_threshold = np.percentile(v_channel, THRESHOLDS["highlight_percentile"])
    highlight_pixels = v_channel >= highlight_threshold
    highlight_area_ratio = float(highlight_pixels.sum()) / len(v_channel)

    v_kurtosis = float(kurtosis(v_channel)) if len(v_channel) > 20 else 0.0
    saturation_std = float(s_channel.std())

    votes = 0
    if highlight_area_ratio >= THRESHOLDS["highlight_area_ratio_min"]:
        votes += 1
    if v_kurtosis >= THRESHOLDS["highlight_kurtosis_min"]:
        votes += 1
    if saturation_std >= THRESHOLDS["saturation_std_min"]:
        votes += 1

    has_glaze_guess = votes >= 2
    confidence_label = {0: "낮음", 1: "낮음", 2: "중간", 3: "높음"}[votes]

    return GlazeEstimate(
        has_glaze_guess=has_glaze_guess,
        confidence_label=confidence_label,
        highlight_area_ratio=round(highlight_area_ratio, 4),
        highlight_kurtosis=round(v_kurtosis, 2),
        saturation_std=round(saturation_std, 2),
        note="규칙 기반 잔정 추정치입니다. THRESHOLDS는 실제 사진으로 재보정이 필요합니다.",
    )

In [ ]:
%%writefile {POTTERY_DIR}/completeness_classifier.py
"""
completeness_classifier.py

도자기가 "완전한 형태"인지 "파편"인지를 외곽선 형태로 1차 판별하는 시작 코드.
자세한 설명은 데스크톱 버전 참고 (동일 로직).
"""

from dataclasses import dataclass

import cv2
import numpy as np

THRESHOLDS = {
    "symmetry_iou_min": 0.7,
    "circularity_min": 0.55,
    "sharp_defect_depth_ratio_max": 0.08,
}


@dataclass
class CompletenessEstimate:
    guess: str
    symmetry_iou: float
    circularity: float
    max_defect_depth_ratio: float
    note: str


def _largest_contour(binary_mask: np.ndarray):
    contours, _ = cv2.findContours(
        binary_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    if not contours:
        return None
    return max(contours, key=cv2.contourArea)


def _symmetry_iou(binary_mask: np.ndarray) -> float:
    ys, xs = np.where(binary_mask)
    center_x = int(round(xs.mean()))
    flipped = np.zeros_like(binary_mask)
    height, width = binary_mask.shape
    for y, x in zip(ys, xs):
        mirrored_x = 2 * center_x - x
        if 0 <= mirrored_x < width:
            flipped[y, mirrored_x] = True
    intersection = np.logical_and(binary_mask, flipped).sum()
    union = np.logical_or(binary_mask, flipped).sum()
    return float(intersection / union) if union > 0 else 0.0


def _circularity(contour, area: float) -> float:
    perimeter = cv2.arcLength(contour, True)
    if perimeter == 0:
        return 0.0
    return float(4 * np.pi * area / (perimeter ** 2))


def _max_defect_depth_ratio(contour, binary_mask: np.ndarray) -> float:
    hull = cv2.convexHull(contour, returnPoints=False)
    if hull is None or len(hull) < 3:
        return 0.0
    try:
        defects = cv2.convexityDefects(contour, hull)
    except cv2.error:
        return 0.0
    if defects is None:
        return 0.0
    diagonal = float(np.hypot(*binary_mask.shape))
    max_depth = max(d[0][3] for d in defects) / 256.0
    return max_depth / diagonal if diagonal > 0 else 0.0


def estimate_completeness(object_mask: np.ndarray) -> CompletenessEstimate:
    binary_mask = np.asarray(object_mask, dtype=bool)
    contour = _largest_contour(binary_mask)

    if contour is None or cv2.contourArea(contour) == 0:
        return CompletenessEstimate(
            guess="판단 보류", symmetry_iou=0.0, circularity=0.0,
            max_defect_depth_ratio=0.0, note="윤곽선을 추출하지 못했습니다.",
        )

    area = cv2.contourArea(contour)
    symmetry_iou = _symmetry_iou(binary_mask)
    circularity = _circularity(contour, area)
    defect_ratio = _max_defect_depth_ratio(contour, binary_mask)

    is_symmetric = symmetry_iou >= THRESHOLDS["symmetry_iou_min"]
    is_smooth = circularity >= THRESHOLDS["circularity_min"]
    has_sharp_defect = defect_ratio > THRESHOLDS["sharp_defect_depth_ratio_max"]

    if is_symmetric and is_smooth and not has_sharp_defect:
        guess = "완전"
    elif has_sharp_defect and not is_symmetric:
        guess = "파편"
    else:
        guess = "판단 보류"

    return CompletenessEstimate(
        guess=guess,
        symmetry_iou=round(symmetry_iou, 3),
        circularity=round(circularity, 3),
        max_defect_depth_ratio=round(defect_ratio, 4),
        note="규칙 기반 잔정 추정치입니다. '판단 보류'는 사람이 확인해야 합니다.",
    )

## 4. 도자기 탐지기 (Grounding DINO + SAM2, 프롬프트만 교체)
`heritage_corrison/models/grounded_sam_detector.py`의 `GroundedSAMDetector`를 그대로 재사용하되,
부식 프롬프트 대신 도자기 프롬프트로 인스턴스를 생성합니다.

In [ ]:
from models.grounded_sam_detector import GroundedSAMDetector

# 생성자가 text_prompt를 직접 받습니다 (부식 검출기와 다른 부분은 이 프롬프트뿐).
POTTERY_TEXT_PROMPT = (
    "ceramic vessel. pottery. clay pot. "
    "ceramic fragment. broken pottery shard."
)

pottery_detector = GroundedSAMDetector(
    text_prompt=POTTERY_TEXT_PROMPT,
    max_regions=1,
)
print('도자기 탐지기 준비 완료. text_prompt =', pottery_detector.text_prompt)

## 5. 통합 분석 함수

In [ ]:
import sys, os

# 세션을 재시작하면 sys.path도 초기화된다. 이전 셀(2번)을 건너뛰고
# 여기로 바로 왔을 경우를 대비해, import 직전에 경로를 다시 확인/추가한다.
PROJECT_ROOT = '/content/drive/MyDrive/kt_bigp'
HERITAGE_REPO = os.path.join(PROJECT_ROOT, 'heritage_corrison')
POTTERY_DIR = os.path.join(PROJECT_ROOT, 'pottery_analysis')

for path_ in (HERITAGE_REPO, POTTERY_DIR):
    if path_ not in sys.path:
        sys.path.insert(0, path_)

expected_file = os.path.join(POTTERY_DIR, 'glaze_detector.py')
if not os.path.exists(expected_file):
    raise FileNotFoundError(
        f'{expected_file} 이 없습니다. 3번 셀(모듈 저장)을 먼저 실행하세요.'
    )

from glaze_detector import estimate_glaze
from completeness_classifier import estimate_completeness
import cv2

def analyze_pottery(image_path: str) -> dict:
    image_bgr = cv2.imread(image_path)
    if image_bgr is None:
        raise FileNotFoundError(image_path)

    regions = pottery_detector.predict(image_path=image_path)
    if not regions:
        return {'status': '탐지 실패', 'reason': 'Grounding DINO + SAM2가 도자기 영역을 찾지 못함'}

    mask = regions[0]['mask']

    glaze = estimate_glaze(image_bgr, mask)
    completeness = estimate_completeness(mask)

    return {
        'status': '성공',
        'completeness': completeness,
        'glaze': glaze,
    }

# 사용 예 (경로를 실제 도자기 사진으로 바꾸세요):
# result = analyze_pottery('/content/drive/MyDrive/kt_bigp/pottery_analysis/sample.jpg')
# print(result)

## 6. 테스트 (사진 업로드해서 바로 실행)

In [ ]:
from google.colab import files

uploaded = files.upload()  # 여기서 도자기 사진 업로드
for filename in uploaded.keys():
    print(f'\n=== {filename} 분석 결과 ===')
    result = analyze_pottery(filename)
    print(result)